# Pipeline Bronze → Silver

**Fonte:** Kaggle - E-Commerce Data (carrie1/ecommerce-data), dataset público, sem restrição de licença para uso acadêmico.

**Objetivo desta etapa:** ler a tabela bronze (dados brutos, sem tratamento) e aplicar limpeza para gerar a camada silver.

In [0]:
df_bronze = spark.table("ecommerce_mvp.bronze.data")
display(df_bronze)
df_bronze.printSchema()
print(f"Total de linhas: {df_bronze.count()}")

In [0]:
from pyspark.sql.functions import col, to_timestamp

df_silver = (
    df_bronze
    .dropDuplicates()                                                # remove linhas totalmente duplicadas (achado da análise de qualidade)
    .filter(col("CustomerID").isNotNull())                          # remove vendas sem cliente identificado
    .filter(col("Quantity") > 0)                                     # remove devoluções (quantidade negativa)
    .filter(col("UnitPrice") > 0)                                    # remove preços inválidos/zerados
    .withColumn("CustomerID", col("CustomerID").cast("int"))         # agora convertido para número
    .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm"))  # parsing do formato visto no print
    .withColumn("TotalPrice", col("Quantity") * col("UnitPrice"))    # coluna com o valor total do produto, será utilizada no cálculo de faturamento
)

df_silver.write.mode("overwrite").saveAsTable("ecommerce_mvp.silver.vendas_tratadas")  # grava a tabela silver atualizada

display(df_silver)
print(f"Linhas na bronze: {df_bronze.count()}")
print(f"Linhas na silver: {df_silver.count()}")

In [0]:
df_silver.write.mode("overwrite").saveAsTable("ecommerce_mvp.silver.vendas_tratadas")

## Resumo da limpeza (Bronze → Silver)
- Linhas brutas: 541.909
- Linhas após limpeza: 397.884 (73,4% mantidas)
- Critérios aplicados:
  - Remoção de registros sem `CustomerID` (venda não identificada)
  - Remoção de `Quantity <= 0` (devoluções/estornos)
  - Remoção de `UnitPrice <= 0` (preços inválidos)
- Transformações: `InvoiceDate` convertido para timestamp; `CustomerID` convertido para inteiro; coluna `TotalPrice` calculada (Quantity × UnitPrice)